In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [13]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
pdf = loader.load()

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(pdf)


In [15]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [16]:
vector_db = Chroma.from_documents(
    embedding=embeddings,
    documents=splitted_data,
)

In [17]:
llm = ChatOpenAI(model="gpt-5")

In [22]:
def get_context(query:str):
    data = vector_db.similarity_search(query=query)
    context = ""
    for chunk in data:
        context += chunk.page_content + "\n"
        
    return {
        "context":context,
        "Question":query,
    }

In [29]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {Question}
""")


In [30]:
rag_chain = get_context | prompt | llm

In [31]:
res = rag_chain.invoke("What is the duration of the course?")
print(res.content)

The course follows a 1-year roadmap. (The first three modules shown total 11 weeks: 4 + 4 + 3.)
